# Lightweight Relational Knowledge Fusion (P2 — ZS-CSD)

Phien ban ZS-CSD cua `relational-knowledge-fusion-with-stanceosaurus.ipynb` — cung y tuong kien truc (rut gon tu **LLM-CRAN**, Niu et al., *MT2-CSD*, arXiv 2506.21053), ap dung cho tap du lieu tieng Trung **ZS-CSD** (`train_data.json`, `dev_data.json`, `test_data.json`; backbone `hfl/chinese-roberta-wwm-ext`).

**Khac biet cot loi so voi `sitpcl-fot-sparsemax-with-zscsd.ipynb` (GRU tuan tu qua SSE_FoT):**
Thay vi ep toan bo hoi thoai qua 1 GRU tuan tu (gia dinh Markov yeu), pipeline nay xay **do thi hoi thoai thuc** (node = luot noi, canh = quan he reply) va dung **GCN/RGCN + multi-hop attention** de tong hop 3 loai tri thuc:

1. **Local Knowledge (LKL):** CNN cua so truot quanh luot dang xet — nam ngu canh cuc bo gan nhat.
2. **Contextual/Structural Knowledge (CRL):** GCN tren do thi reply-chain thuc — nam quan he toan cuc, khong bi "quen" nhu GRU khi hoi thoai dai.
3. **Discourse-Relation Knowledge (RGCN):** quan he logic giua cac luot (contrastive/causal/succession/summary), gan nhan bang **bo phan loai rule-based dua discourse marker tieng Trung** (但是/不过/虽然 — tuong phan; 因为/所以/由于 — nhan qua; 然后/接着/首先 — tiep dien) — thay cho GPT-4 trong KAM goc cua LLM-CRAN, de chay offline mien phi tren Kaggle. Day la surrogate re tien, yeu hon LLM thuc, co the nang cap sau neu co API.

**Giu nguyen de so sanh cong bang voi `sitpcl-fot-sparsemax-with-zscsd.ipynb`:**
- Cung backbone `hfl/chinese-roberta-wwm-ext`, cung prompt cloze-style `[CLS]在话语"{sen}"中，用户{speaker}对[SEP]{target}[SEP]的立场为[MASK][SEP]` va cung cach trich bieu dien tai vi tri `[MASK]` (khong mean-pooling) cho moi luot.
- Cung input: chi `train_data.json`/`dev_data.json`/`test_data.json` — ZS-CSD khong co dataset "evidence" bo sung nao, nen khong co van de opt-in evidence injection nhu ben Stanceosaurus.
- Cung ham mat mat: `CrossEntropyLoss` thuan (khong Focal, khong class-weight) — **giong dung** baseline SITPCL-FoT+Sparsemax hien tai, de su khac biet ket qua chi phan anh kien truc (GCN/RGCN) khong bi tron voi thay doi loss.
- Cung bang doi sanh theo chuan bai bao goc (Mixed / Noun-phrase / Claim targets) va cung chinh sach Max Epochs=30 + Early Stopping patience=5 vua ap dung cho toan bo 6 pipeline con lai.

**Luu y:** day la huong kien truc moi, rui ro/do phuc tap trien khai cao hon SITPCL-FoT. Batch size giam xuong 8 voi gradient accumulation 2 buoc = effective batch 16, bang dung baseline, vi moi hoi thoai gio ton nhieu bo nho hon (CNN+GCN+RGCN).

**Yeu cau Kaggle Input:** chi can dataset chua `train_data.json`, `dev_data.json`, `test_data.json` — giong het `sitpcl-fot-sparsemax-with-zscsd.ipynb`.

**Che do chay thu nhanh:** doi `config.debug = True` o Cell 2 de chay ~30 mau truoc khi treo may qua dem (`debug = False`).

In [ ]:
!pip install -q python-box transformers scikit-learn tqdm

import os
import glob

# Tự động quét tìm dataset trong /kaggle/input/
found_train = glob.glob("/kaggle/input/**/train_data.json", recursive=True) + \
              glob.glob("./**/train_data.json", recursive=True) + \
              glob.glob("D:/KLTN/**/train_data.json", recursive=True)
if not found_train:
    raise FileNotFoundError("Không tìm thấy train_data.json! Hãy kiểm tra lại dataset ở tab Input bên phải.")

DATA_DIR = os.path.dirname(found_train[0])
print(f"--> Đã kết nối dữ liệu tại: {DATA_DIR}")

In [ ]:
from box import Box
import torch
from datetime import datetime

config = Box({
    # Đường dẫn dữ liệu — giống hệt sitpcl-fot-sparsemax-with-zscsd.ipynb
    "train_path": os.path.join(DATA_DIR, "train_data.json"),
    "dev_path": os.path.join(DATA_DIR, "dev_data.json"),
    "test_path": os.path.join(DATA_DIR, "test_data.json"),

    # Backbone PLM
    "bert_dir": "hfl/chinese-roberta-wwm-ext",

    # Siêu tham số huấn luyện
    "epoch_size": 30,          # Trần epoch cao, dừng thật bằng Early Stopping
    "early_stop_patience": 5,  # Dừng sớm nếu Dev Macro-F1 không cải thiện sau 5 epoch liên tiếp
    "batchsize": 8,            # Mỗi hội thoại tốn nhiều bộ nhớ hơn (CNN+GCN+RGCN) nên giảm batch
    "accum_steps": 2,          # Effective batch = 16, bằng đúng baseline SITPCL-FoT+Sparsemax
    "bert_lr": 1e-5,
    "gnn_lr": 1e-4,            # LR riêng cho CNN/GCN/RGCN/fusion/classifier
    "weight_decay": 1e-6,
    "warmup_steps": 0,
    "adam_epsilon": 1e-8,
    "max_grad_norm": 1.0,

    # Siêu tham số kiến trúc Relational Knowledge Fusion
    "cnn_kernel": 3,
    "num_relations_logical": 4,   # contrastive, causal, succession, summary
    "multihop_iters": 2,
    "multihop_lambda": 0.2,
    "num_classes": 3,             # Favor, Against, Neutral

    "seed": 1234,
    "cuda_index": 0,

    # Chế độ thử nghiệm
    "debug": False,               # True: Test 30 mẫu trong vài chục giây | False: Chạy full qua đêm
    "model_type": "Relational_Knowledge_Fusion",
    "time": datetime.now().strftime("%Y%m%d_%H%M%S")
})

config.device = torch.device(f"cuda:{config.cuda_index}" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị: {config.device} | Mô hình: {config.model_type} | Debug: {config.debug}")
print(f"Max Epochs: {config.epoch_size} (early stop patience={config.early_stop_patience}) | Batch: {config.batchsize}x{config.accum_steps}")

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# --- Bộ phân loại quan hệ discourse rule-based tiếng Trung ---
# Surrogate OFFLINE của bước KAM trong LLM-CRAN gốc (vốn dùng GPT-4 để gán quan hệ).
# Thay bằng gọi LLM thật (GPT-4/Qwen/Llama) nếu có API sẽ chính xác hơn — ghi rõ trong báo cáo.
CONTRASTIVE_MARKERS = ['但是', '不过', '虽然', '然而', '尽管', '可是', '却']
CAUSAL_MARKERS = ['因为', '所以', '由于', '因此', '以致', '故而']
SUCCESSION_MARKERS = ['然后', '接着', '首先', '其次', '最后', '同时', '而且', '并且', '还有']
RELATION_NAMES = ['contrastive', 'causal', 'succession', 'summary']  # fallback = 'summary'
RELATION_TO_ID = {name: i for i, name in enumerate(RELATION_NAMES)}

def classify_discourse_relation(prev_text, curr_text):
    """Heuristic rẻ tiền dựa discourse marker ở đầu câu hiện tại. Tiếng Trung không có khoảng trắng
    rõ giữa từ nên so khớp theo chuỗi con trong 12 ký tự đầu, thay vì tách từ theo khoảng trắng như
    bản tiếng Anh ở Stanceosaurus."""
    t = (curr_text or '').strip()
    prefix = t[:12]
    for m in CONTRASTIVE_MARKERS:
        if m in prefix:
            return RELATION_TO_ID['contrastive']
    for m in CAUSAL_MARKERS:
        if m in prefix:
            return RELATION_TO_ID['causal']
    for m in SUCCESSION_MARKERS:
        if m in prefix:
            return RELATION_TO_ID['succession']
    return RELATION_TO_ID['summary']

In [ ]:
import json
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        return self.data[index]

class DataProcessorP2:
    """Giữ nguyên prompt cloze-style và cách trích vị trí [MASK] giống hệt DataProcessor của
    sitpcl-fot-sparsemax-with-zscsd.ipynb — chỉ khác ở việc gán thêm cạnh đồ thị (edges) và
    quan hệ discourse (relations) cho mỗi hội thoại để nuôi GCN/RGCN."""
    def __init__(self, config):
        self.tokenizer = AutoTokenizer.from_pretrained(config.bert_dir)
        self.config = config
        self.mask_token_id = self.tokenizer.mask_token_id

    def read_data(self, mode):
        file_path = getattr(self.config, f"{mode}_path")
        content = json.load(open(file_path, 'r', encoding='utf-8'))

        if getattr(self.config, 'debug', False):
            content = content[:30]

        res = []
        for line in tqdm(content, desc=f'Processing dialogues for {mode}'):
            res.append(self.parse_dialogue(line))
        return res

    def parse_dialogue(self, dialogue):
        target = dialogue["target"]
        speakers = dialogue["speakers"]
        sentences_raw = dialogue["sentences"]
        n = len(sentences_raw)

        input_ids_list = []
        mask_positions = []
        for idx, sen in enumerate(sentences_raw):
            prompt = f'[CLS]在话语"{sen}"中，用户{speakers[idx]}对[SEP]{target}[SEP]的立场为[MASK][SEP]'
            tokens = self.tokenizer.tokenize(prompt)
            ids = self.tokenizer.convert_tokens_to_ids(tokens)
            mpos = ids.index(self.mask_token_id) if self.mask_token_id in ids else max(len(ids) - 2, 0)
            input_ids_list.append(ids)
            mask_positions.append(mpos)

        # Đồ thị reply-chain tuyến tính (không có branching info trong dữ liệu ZS-CSD)
        edges = [(i - 1, i) for i in range(1, n)]
        relations = [classify_discourse_relation(sentences_raw[i - 1], sentences_raw[i]) for i in range(1, n)]

        return {
            'input_ids': input_ids_list,
            'mask_positions': mask_positions,
            'edges': edges,
            'relations': relations,
            'label': dialogue['label'],
            'id': dialogue['id'],
            'target_type': dialogue.get('target_type', 'n')
        }

    def collate_fn(self, batch):
        all_ids, mask_indices = [], []
        dia_idx, edges_list, relations_list, labels, doc_ids, target_types = [], [], [], [], [], []

        curr = 0
        for item in batch:
            n = len(item['input_ids'])
            dia_idx.append((curr, curr + n))
            curr += n
            all_ids.extend(item['input_ids'])
            mask_indices.extend(item['mask_positions'])
            edges_list.append(item['edges'])
            relations_list.append(item['relations'])
            labels.append(item['label'])
            doc_ids.append(item['id'])
            target_types.append(item['target_type'])

        max_len = max(len(ids) for ids in all_ids)
        padded_ids = [ids + [0] * (max_len - len(ids)) for ids in all_ids]
        padded_masks = [[1] * len(ids) + [0] * (max_len - len(ids)) for ids in all_ids]

        return {
            'input_ids': torch.tensor(padded_ids, dtype=torch.long).to(self.config.device),
            'input_masks': torch.tensor(padded_masks, dtype=torch.long).to(self.config.device),
            'mask_indices': torch.tensor(mask_indices, dtype=torch.long).to(self.config.device),
            'dia_idx': dia_idx,
            'edges': edges_list,
            'relations': relations_list,
            'label': torch.tensor(labels, dtype=torch.long).to(self.config.device),
            'doc_id': doc_ids,
            'target_type': target_types
        }

    def get_data(self):
        train_dataset = MyDataset(self.read_data('train'))
        dev_dataset = MyDataset(self.read_data('dev'))
        test_dataset = MyDataset(self.read_data('test'))

        train_loader = DataLoader(train_dataset, batch_size=self.config.batchsize, shuffle=True, collate_fn=self.collate_fn)
        dev_loader = DataLoader(dev_dataset, batch_size=self.config.batchsize, shuffle=False, collate_fn=self.collate_fn)
        test_loader = DataLoader(test_dataset, batch_size=self.config.batchsize, shuffle=False, collate_fn=self.collate_fn)
        return train_loader, dev_loader, test_loader

In [ ]:
from transformers import AutoModel

class LocalKnowledgeLayer(nn.Module):
    """CNN 1D cửa sổ trượt quanh lượt đang xét — nắm ngữ cảnh cục bộ gần nhất (LKL trong LLM-CRAN)."""
    def __init__(self, dim, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=pad)
        self.conv2 = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=pad)
        self.act = nn.GELU()

    def forward(self, H):          # H: (N, D)
        x = H.T.unsqueeze(0)       # (1, D, N)
        x = self.act(self.conv1(x))
        x = self.conv2(x)
        return x.squeeze(0).T      # (N, D)

class GCNLayer(nn.Module):
    """GCN 2 lớp trên đồ thị reply-chain — thay thế GRU tuần tự (CRL trong LLM-CRAN)."""
    def __init__(self, dim):
        super().__init__()
        self.w0 = nn.Linear(dim, dim)
        self.w1 = nn.Linear(dim, dim)
        self.act = nn.GELU()

    def forward(self, H, adj_norm):
        h = self.act(adj_norm @ self.w0(H))
        h = adj_norm @ self.w1(h)
        return h

class RGCNLayer(nn.Module):
    """RGCN 2 lớp trên đồ thị quan hệ discourse có kiểu cạnh (LRL trong LLM-CRAN, rút gọn 2 lớp)."""
    def __init__(self, dim, num_relations):
        super().__init__()
        self.num_relations = num_relations
        self.w_rel = nn.ModuleList([nn.Linear(dim, dim) for _ in range(num_relations)])
        self.w_self = nn.Linear(dim, dim)
        self.act = nn.GELU()

    def forward(self, H, rel_adj_list):
        out = self.w_self(H)
        for r, adj_r in enumerate(rel_adj_list):
            out = out + self.w_rel[r](adj_r @ H)
        return self.act(out)

def build_adjacency(n, edges, device):
    """Đồ thị vô hướng có self-loop, chuẩn hoá kiểu GCN: D^-1/2 (A+I) D^-1/2."""
    A = torch.eye(n, device=device)
    for i, j in edges:
        A[i, j] = 1.0
        A[j, i] = 1.0
    deg = A.sum(dim=1)
    deg_inv_sqrt = torch.pow(deg.clamp(min=1e-6), -0.5)
    D_inv_sqrt = torch.diag(deg_inv_sqrt)
    return D_inv_sqrt @ A @ D_inv_sqrt

def build_relational_adjacency(n, edges, relations, num_relations, device):
    mats = [torch.zeros(n, n, device=device) for _ in range(num_relations)]
    for (i, j), r in zip(edges, relations):
        mats[r][i, j] = 1.0
        mats[r][j, i] = 1.0
    normed = []
    for A in mats:
        deg = A.sum(dim=1)
        deg_inv = torch.where(deg > 0, 1.0 / deg, torch.zeros_like(deg))
        normed.append(torch.diag(deg_inv) @ A)
    return normed

class MultiHopAttentionFusion(nn.Module):
    """Hợp nhất 3 luồng tri thức (Hl, Hs, Hr) có điều kiện theo vector lượt cuối (anchor), lặp p vòng."""
    def __init__(self, dim, n_iters=2, lam=0.2):
        super().__init__()
        self.n_iters = n_iters
        self.lam = lam
        self.norm = nn.LayerNorm(dim)
        self.proj = nn.ModuleList([nn.Linear(dim, 1) for _ in range(3)])

    def forward(self, Hl, Hs, Hr, anchor_idx):
        streams = [Hl, Hs, Hr]
        for _ in range(self.n_iters):
            new_streams = []
            for k, Hk in enumerate(streams):
                anchor = Hk[anchor_idx].unsqueeze(0)
                scores = self.proj[k](Hk * anchor).squeeze(-1)
                c = torch.sigmoid(scores).unsqueeze(-1)
                update = self.norm(c * Hk)
                new_streams.append(Hk + self.lam * update)
            streams = new_streams
        return torch.cat([s[anchor_idx] for s in streams], dim=-1)   # (3*D,)

class RelationalFusionModelZSCSD(nn.Module):
    """Giống RelationalFusionModel bên Stanceosaurus, chỉ đổi backbone sang chinese-roberta-wwm-ext
    và trích biểu diễn mỗi lượt tại vị trí [MASK] (giống đúng SITPCL-FoT+Sparsemax) thay vì mean-pooling,
    để giữ công bằng về cách biểu diễn câu giữa 2 pipeline ZS-CSD."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.bert = AutoModel.from_pretrained(config.bert_dir)
        dim = self.bert.config.hidden_size
        self.lkl = LocalKnowledgeLayer(dim, kernel_size=config.cnn_kernel)
        self.crl = GCNLayer(dim)
        self.lrl = RGCNLayer(dim, num_relations=config.num_relations_logical)
        self.fusion = MultiHopAttentionFusion(dim, n_iters=config.multihop_iters, lam=config.multihop_lambda)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(dim * 3, config.num_classes))
        # Thuần CrossEntropyLoss, không Focal/class-weight -- giống đúng baseline SITPCL-FoT+Sparsemax
        # để khác biệt kết quả chỉ phản ánh kiến trúc (GCN/RGCN), không trộn với thay đổi hàm mất mát.
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input_ids, input_masks, mask_indices, dia_idx, edges, relations, label=None, **kwargs):
        out = self.bert(input_ids=input_ids, attention_mask=input_masks).last_hidden_state
        batch_idx = torch.arange(out.size(0), device=out.device)
        all_H = out[batch_idx, mask_indices, :]   # Biểu diễn [MASK] của mỗi lượt

        logits_list = []
        for d_id, (st, ed) in enumerate(dia_idx):
            H = all_H[st:ed]
            n = H.size(0)
            e, r = edges[d_id], relations[d_id]

            Hl = self.lkl(H)
            if n > 1:
                adj = build_adjacency(n, e, out.device)
                Hs = self.crl(H, adj)
                rel_adjs = build_relational_adjacency(n, e, r, self.config.num_relations_logical, out.device)
                Hr = self.lrl(H, rel_adjs)
            else:
                Hs, Hr = H, H

            fused = self.fusion(Hl, Hs, Hr, anchor_idx=n - 1)
            logits_list.append(fused)

        stance_features = torch.stack(logits_list)
        logits = self.classifier(stance_features)

        loss = None
        if label is not None:
            loss = self.criterion(logits, label)
        return loss, logits

In [ ]:
import pandas as pd
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score

class Trainer:
    def __init__(self, config):
        self.config = config
        set_seed(self.config.seed)

        prefix = "debug_" if self.config.debug else ""
        self.save_dir = f"./result/{prefix}{self.config.model_type}_{self.config.time}/save/"
        self.pred_file = f"./result/{prefix}{self.config.model_type}_{self.config.time}/pred_{self.config.seed}.csv"
        os.makedirs(self.save_dir, exist_ok=True)
        os.makedirs(os.path.dirname(self.pred_file), exist_ok=True)

        self.best_dev_f1 = -1.0
        self.patience_counter = 0

    def calc_group_metrics(self, y_true, y_pred):
        if len(y_true) == 0:
            return 0.0, 0.0, 0.0, 0.0
        f1_per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        return f1_per_class[0] * 100, f1_per_class[1] * 100, f1_per_class[2] * 100, macro_f1 * 100

    def print_paper_comparison_table(self, trues, preds, target_types, epoch_info="BEST TEST"):
        trues = np.array(trues)
        preds = np.array(preds)
        target_types = np.array([str(t).lower().strip() for t in target_types])

        mix_fav, mix_aga, mix_neu, mix_all = self.calc_group_metrics(trues, preds)
        noun_mask = (target_types == 'n')
        noun_fav, noun_aga, noun_neu, noun_all = self.calc_group_metrics(trues[noun_mask], preds[noun_mask])
        claim_mask = (target_types == 'c')
        clm_fav, clm_aga, clm_neu, clm_all = self.calc_group_metrics(trues[claim_mask], preds[claim_mask])

        total_acc = accuracy_score(trues, preds) * 100

        print(f"\n{'='*105}")
        print(f"{f'BẢNG KẾT QUẢ ĐỐI SÁNH THEO CHUẨN BÀI BÁO (TABLE 2) [{epoch_info}]':^105}")
        print(f"{'='*105}")
        print(f"{'METHOD':<20} | {'Mixed targets':^26} | {'Noun-phrase targets':^26} | {'Claim targets':^26}")
        print(f"{'':<20} | {'Favor':>5} {'Against':>7} {'Neutral':>7} {'All':>5} | {'Favor':>5} {'Against':>7} {'Neutral':>7} {'All':>5} | {'Favor':>5} {'Against':>7} {'Neutral':>7} {'All':>5}")
        print(f"{'-'*21}+{'-'*28}+{'-'*28}+{'-'*27}")
        print(f"{'Paper SITPCL':<20} | 41.03   41.34   49.07 43.81 | 40.99   47.14   54.49 47.54 | 41.06   37.34   46.02 41.47")
        print(f"{self.config.model_type:<20} | {mix_fav:5.2f} {mix_aga:7.2f} {mix_neu:7.2f} {mix_all:5.2f} | {noun_fav:5.2f} {noun_aga:7.2f} {noun_neu:7.2f} {noun_all:5.2f} | {clm_fav:5.2f} {clm_aga:7.2f} {clm_neu:7.2f} {clm_all:5.2f}")
        print(f"{'='*105}")
        print(f"  * Số mẫu: Total={len(trues)} | Noun-phrase={np.sum(noun_mask)} | Claim={np.sum(claim_mask)} | Accuracy={total_acc:.2f}%")
        print(f"{'='*105}\n")

    def train_iter(self):
        self.model.train()
        running_loss = 0.0
        self.optimizer.zero_grad()
        n_batches = len(self.trainLoader)
        for step, data in enumerate(tqdm(self.trainLoader, desc="Train Iter")):
            loss, _ = self.model(**data)
            loss = loss / self.config.accum_steps
            loss.backward()
            running_loss += loss.item() * self.config.accum_steps

            if (step + 1) % self.config.accum_steps == 0 or (step + 1) == n_batches:
                nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()
        return running_loss / n_batches

    def evaluate_iter(self, mode=None):
        self.model.eval()
        val_loss = 0.0
        seq_preds, seq_trues, doc_id_lst, target_types = [], [], [], []
        dataLoader = self.devLoader if mode == 'dev' else self.testLoader

        with torch.no_grad():
            for data in tqdm(dataLoader, desc=f"Eval {mode}"):
                loss, seq_output = self.model(**data)
                val_loss += loss.item()
                seq_output = np.argmax(seq_output.detach().cpu().numpy(), -1)
                labels = data['label'].detach().cpu().numpy().reshape(-1)

                seq_preds.extend(seq_output)
                seq_trues.extend(labels)
                doc_id_lst.extend(data['doc_id'])
                target_types.extend(data['target_type'])

        val_loss /= len(dataLoader)
        macro_f1 = f1_score(seq_trues, seq_preds, average='macro', zero_division=0)

        if mode == 'test':
            pd.DataFrame({'doc_id': doc_id_lst, 'true': seq_trues, 'pred': seq_preds, 'target_type': target_types}).to_csv(self.pred_file, index=False)

        return macro_f1, val_loss, seq_trues, seq_preds, target_types

    def setup_optimization(self):
        backbone_params = list(self.model.bert.parameters())
        other_params = [p for n, p in self.model.named_parameters() if not n.startswith('bert.')]
        self.optimizer = AdamW([
            {'params': backbone_params, 'lr': float(self.config.bert_lr), 'weight_decay': float(self.config.weight_decay)},
            {'params': other_params, 'lr': float(self.config.gnn_lr), 'weight_decay': float(self.config.weight_decay)}
        ], eps=float(self.config.adam_epsilon))
        total_steps = (len(self.trainLoader) // self.config.accum_steps) * (1 if self.config.debug else self.config.epoch_size)
        self.scheduler = get_linear_schedule_with_warmup(self.optimizer, num_warmup_steps=self.config.warmup_steps, num_training_steps=max(total_steps, 1))

    def run(self):
        print("--> Đang nạp dữ liệu...")
        self.trainLoader, self.devLoader, self.testLoader = DataProcessorP2(self.config).get_data()

        print(f"--> Khởi tạo mô hình {self.config.model_type}...")
        self.model = RelationalFusionModelZSCSD(self.config).to(self.config.device)
        self.setup_optimization()

        epochs = 1 if self.config.debug else self.config.epoch_size
        patience = self.config.early_stop_patience
        print(f"--> BẮT ĐẦU HUẤN LUYỆN (Chế độ: {'DEBUG' if self.config.debug else 'FULL DATA'})...\n")

        for epoch in range(epochs):
            train_loss = self.train_iter()
            dev_macro_f1, dev_loss, _, _, _ = self.evaluate_iter(mode='dev')
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {dev_loss:.4f} | Val Macro F1: {100 * dev_macro_f1:.2f}%")

            if dev_macro_f1 > self.best_dev_f1:
                self.best_dev_f1 = dev_macro_f1
                self.patience_counter = 0
                test_macro_f1, test_loss, trues, preds, types = self.evaluate_iter(mode='test')
                print(f"\n>>> [Epoch {epoch+1}] TÌM THẤY BEST MODEL MỚI! (Dev F1: {100*dev_macro_f1:.2f}%) <<<")
                self.print_paper_comparison_table(trues, preds, types, epoch_info=f"Epoch {epoch+1}")
                torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'best_model.pth'))
            else:
                self.patience_counter += 1
                print(f"    ⏳ [Không cải thiện] {self.patience_counter}/{patience} epoch liên tiếp (Best Dev F1: {100*self.best_dev_f1:.2f}%)")
                if self.patience_counter >= patience:
                    print(f"\n>>> 🛑 [EARLY STOPPING] Dừng tại epoch {epoch+1} vì Dev Macro-F1 không cải thiện sau {patience} epoch liên tiếp. <<<")
                    break

        print("\n Hoàn thành huấn luyện!")

In [ ]:
trainer = Trainer(config)
trainer.run()